# Connectome-Constrained Ring Attractor Simulation
## Comparative Dynamics: *Drosophila* vs *Aedes aegypti*

Rate-based leaky-integrator model of the heading direction circuit.

$$\tau_k \cdot \frac{dr_i}{dt} = -r_i + \phi\!\left(\sum_j W_{ij} \cdot r_j + I_{\text{ext},i}\right)$$


In [ ]:
# ================================================================
# Google Colab Setup
# ================================================================
import os

IN_COLAB = 'google.colab' in str(globals().get('get_ipython', lambda: '')())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/aedes_ring_attractor'
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR, exist_ok=True)
        print(f'Created {DATA_DIR} — upload connectome_data.pkl there')
else:
    DATA_DIR = 'data'

DATA_PATH = os.path.join(DATA_DIR, 'connectome_data.pkl')
print(f'Data: {DATA_PATH} (exists={os.path.exists(DATA_PATH)})')


In [ ]:
# ================================================================
# Inlined cx_model package (for self-contained Colab execution)
# ================================================================

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# activation.py
# ============================================================

"""
Activation Functions
====================
Nonlinear transfer functions that map synaptic drive to firing rate.

φ(drive) → firing rate ∈ [0, r_max]
"""

import numpy as np


def phi_threshold_linear(x, theta=0.0, r_max=100.0):
    """
    Threshold-linear activation with hard saturation.

    φ(x) = clip(x − θ, 0, r_max)

    Parameters
    ----------
    x : np.ndarray
        Total drive to each neuron.
    theta : float
        Firing threshold.
    r_max : float
        Maximum firing rate (Hz).

    Returns
    -------
    np.ndarray
        Firing rates in [0, r_max].
    """
    return np.clip(x - theta, 0.0, r_max)


def phi_sigmoid(x, theta=0.0, r_max=100.0, beta=0.1):
    """
    Sigmoid activation (smooth saturation).

    φ(x) = r_max / (1 + exp(−β(x − θ)))

    Parameters
    ----------
    x : np.ndarray
        Total drive to each neuron.
    theta : float
        Inflection point (rate = r_max/2 when drive = θ).
    r_max : float
        Maximum firing rate (Hz).
    beta : float
        Steepness. Smaller → more graded; larger → sharper.

    Returns
    -------
    np.ndarray
        Firing rates in [0, r_max].
    """
    return r_max / (1.0 + np.exp(-beta * (x - theta)))


# ============================================================
# data_loader.py
# ============================================================

"""
Connectome Data Loader
======================
Load and preprocess EPG–Δ7 connectivity from connectome_data.pkl.
Supports synapse-count thresholding and row-sum renormalization.
"""

import pickle
import numpy as np
import pandas as pd


def load_connectome(filepath, syn_threshold=0):
    """
    Load connectome data and optionally apply a synapse-count threshold.

    Parameters
    ----------
    filepath : str
        Path to connectome_data.pkl.
    syn_threshold : int
        Minimum synapse count for a connection to be retained.
        Connections with fewer synapses are zeroed out before
        renormalization.  Set to 0 to keep all connections.

    Returns
    -------
    data : dict
        All original keys plus thresholded 'matrix_norm_fly' and
        'matrix_norm_aedes' if syn_threshold > 0.
    """
    with open(filepath, 'rb') as f:
        data = pickle.load(f)

    if syn_threshold > 0:
        for species in ['fly', 'aedes']:
            raw_key = f'matrix_{species}'
            norm_key = f'matrix_norm_{species}'
            if raw_key in data:
                data[norm_key] = _threshold_and_renorm(
                    data[raw_key], syn_threshold, species
                )
    return data


def _threshold_and_renorm(matrix_raw, threshold, label=''):
    """Zero connections below threshold, then row-normalise."""
    mat = matrix_raw.copy()
    before = (mat.values > 0).sum()
    mat.values[mat.values < threshold] = 0
    after = (mat.values > 0).sum()
    print(f'  {label}: syn >= {threshold} → '
          f'{before} → {after} connections '
          f'({before - after} removed, {(before-after)/max(before,1)*100:.1f}%)')
    row_sums = mat.values.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return pd.DataFrame(
        mat.values / row_sums,
        index=mat.index, columns=mat.columns
    )


# -- Default neuron orderings -----------------------------------------------

DELTA7_ORDER_FLY = [
    'L8R1R9', 'L7R3', 'L7R2', 'L6R4', 'L6R3',
    'L5R4', 'L4R6', 'L4R5', 'L3R6', 'L3R7', 'L2R7', 'L1L9R8',
]

DELTA7_ORDER_AEDES = [
    'L1L8L9R1', 'L8R1R9', 'L7R2', 'L7R3', 'L6R3R4', 'L5R4',
    'L4L5R4R5', 'L4R5', 'L3L4R6', 'L3R7', 'L2R7', 'L1L9R8', 'L1R1R8R9',
]

EB_RING_ORDER = [
    'R1', 'L8', 'R2', 'L7', 'R3', 'L6', 'R4', 'L5',
    'R5', 'L4', 'R6', 'L3', 'R7', 'L2', 'R8', 'L1',
]


def get_species_data(data, species='fly'):
    """
    Extract sorted IDs, labels, and orderings for one species.

    Parameters
    ----------
    data : dict
        From load_connectome().
    species : str
        'fly' or 'aedes'.

    Returns
    -------
    info : dict with keys
        matrix_norm, sorted_ids_eb, sorted_ids_pb,
        id_to_type, id_to_pb, delta7_order
    """
    if species == 'fly':
        return {
            'matrix_norm': data['matrix_norm_fly'],
            'sorted_ids_eb': data['sorted_ids_fly_eb'],
            'sorted_ids_pb': data['sorted_body_ids_fly'],
            'id_to_type': data['id_to_type_fly'],
            'id_to_pb': data['id_to_pb_fly'],
            'delta7_order': DELTA7_ORDER_FLY,
        }
    else:
        return {
            'matrix_norm': data['matrix_norm_aedes'],
            'sorted_ids_eb': data['sorted_ids_aedes_eb'],
            'sorted_ids_pb': data['sorted_ids_aedes'],
            'id_to_type': data['id_to_type_aedes'],
            'id_to_pb': data['id_to_pb_aedes'],
            'delta7_order': DELTA7_ORDER_AEDES,
        }


# ============================================================
# weights.py
# ============================================================

"""
Weight Matrix Construction
==========================
Build simulation weight matrices from connectome data using
4-block (E→E, E→I, I→E, I→I) structure with sign assignment.
"""

import numpy as np

# Dale's-law sign convention for each block
BLOCK_SIGNS = {'EE': +1, 'IE': -1, 'EI': +1, 'II': -1}


def build_weight_matrix(matrix_norm, sorted_ids_eb, id_to_type, id_to_pb,
                        gain, delta7_order):
    """
    Build the full signed weight matrix W from connectome data.

    Neurons are ordered EPG-first (EB ring order), then Δ7 (custom order).
    W[i, j] = connection from neuron j → neuron i.

    Parameters
    ----------
    matrix_norm : pd.DataFrame
        Normalised connectivity matrix (neuron_id × neuron_id).
    sorted_ids_eb : list
        Neuron IDs sorted by EB wedge position.
    id_to_type : dict
        neuron_id → 'EPG' or 'Delta7'.
    id_to_pb : dict
        neuron_id → PB glomerulus label.
    gain : float
        Global synaptic weight scaling factor.
    delta7_order : list
        Custom ordering for Δ7 PB glomeruli.

    Returns
    -------
    W : np.ndarray (N_total, N_total)
    sim_uids : list
        Neuron IDs in simulation order.
    N_E : int
        Number of EPG neurons.
    type_ranges : dict
        {'EPG': (0, N_E), 'Delta7': (N_E, N_total)}.
    """
    e_ids = [uid for uid in sorted_ids_eb if id_to_type[uid] == 'EPG']
    i_ids = [uid for uid in sorted_ids_eb if id_to_type[uid] == 'Delta7']

    # Sort Δ7 by custom PB order
    pb_rank = {pb: r for r, pb in enumerate(delta7_order)}
    i_ids = sorted(i_ids, key=lambda u: (pb_rank.get(id_to_pb.get(u, '?'), 999), u))

    sim_uids = e_ids + i_ids
    N_E = len(e_ids)
    N_total = len(sim_uids)
    type_ranges = {'EPG': (0, N_E), 'Delta7': (N_E, N_total)}

    # Extract sub-matrices and assemble with signs
    EtoE = matrix_norm.loc[e_ids, e_ids].values.T
    ItoE = matrix_norm.loc[i_ids, e_ids].values.T
    EtoI = matrix_norm.loc[e_ids, i_ids].values.T
    ItoI = matrix_norm.loc[i_ids, i_ids].values.T

    W = np.block([
        [gain * BLOCK_SIGNS['EE'] * EtoE, gain * BLOCK_SIGNS['IE'] * ItoE],
        [gain * BLOCK_SIGNS['EI'] * EtoI, gain * BLOCK_SIGNS['II'] * ItoI],
    ])

    return W, sim_uids, N_E, type_ranges


def find_epg_indices(sim_uids, id_to_type, id_to_pb, N_E, pb_labels):
    """
    Find EPG neuron indices matching given PB glomerulus labels.

    Parameters
    ----------
    sim_uids : list
        Neuron IDs in simulation order.
    id_to_type, id_to_pb : dict
        Mapping dicts.
    N_E : int
        Number of EPG neurons.
    pb_labels : list of str
        e.g. ['L4'] or ['R1'].

    Returns
    -------
    list of int
        Indices into the rate vector.
    """
    if pb_labels == ['none'] or pb_labels is None:
        return []
    return [
        i for i, uid in enumerate(sim_uids)
        if i < N_E
        and id_to_type.get(uid) == 'EPG'
        and id_to_pb.get(uid) in pb_labels
    ]


def extract_4blocks(matrix_norm, sorted_ids, id_to_type, id_to_pb,
                    i_order=None):
    """
    Split connectivity into four E–I blocks for visualisation.

    Returns
    -------
    blocks : dict  {'E→E', 'E→I', 'I→E', 'I→I'} → pd.DataFrame
    row_labels, col_labels : dict → list of str
    """
    e_ids = [u for u in sorted_ids if id_to_type.get(u) == 'EPG']
    i_ids = [u for u in sorted_ids if id_to_type.get(u) == 'Delta7']
    if i_order:
        rank = {pb: r for r, pb in enumerate(i_order)}
        i_ids = sorted(i_ids, key=lambda u: (rank.get(id_to_pb.get(u, '?'), 999), u))

    el = [id_to_pb.get(u, '?') for u in e_ids]
    il = [id_to_pb.get(u, '?') for u in i_ids]

    blocks = {
        'E→E': matrix_norm.loc[e_ids, e_ids],
        'I→E': matrix_norm.loc[i_ids, e_ids],
        'E→I': matrix_norm.loc[e_ids, i_ids],
        'I→I': matrix_norm.loc[i_ids, i_ids],
    }
    rl = {'E→E': el, 'I→E': il, 'E→I': el, 'I→I': il}
    cl = {'E→E': el, 'I→E': el, 'E→I': il, 'I→I': il}
    return blocks, rl, cl


# ============================================================
# simulation.py
# ============================================================

"""
Ring Attractor Simulation Engine
=================================
Rate-based leaky integrator for heading circuit dynamics.

Model equation (per neuron i of type k):
    τ_k · dr_i/dt = −r_i + φ(Σ_j W_ij · r_j + I_ext_i)

Based on Kakaria & de Bivort 2017 and Pisokas et al. 2020.
"""

import numpy as np
# from .activation import phi_threshold_linear  # inlined above


def simulate_ring_attractor(W, N_total, type_ranges, tau_dict,
                            I_ext_func, T=10.0, dt=0.01,
                            phi_func=None, phi_kwargs=None,
                            r0=None, noise_sigma=0.0):
    """
    Simulate ring attractor dynamics via Euler integration.

    Parameters
    ----------
    W : np.ndarray (N_total, N_total)
        Weight matrix.  W[i, j] = connection from j → i.
    N_total : int
        Number of neurons.
    type_ranges : dict
        Cell type → (start_idx, end_idx).
    tau_dict : dict
        Cell type → membrane time constant (seconds).
    I_ext_func : callable(t, N_total) → np.ndarray
        External stimulus at time t.
    T, dt : float
        Duration and timestep (seconds).
    phi_func : callable, optional
        Activation function (default: phi_threshold_linear).
    phi_kwargs : dict, optional
        Keyword args for phi_func (default: {'theta': 0, 'r_max': 100}).
    r0 : np.ndarray, optional
        Initial rates (default: zeros).
    noise_sigma : float
        Gaussian noise σ added to drive each step.

    Returns
    -------
    times : np.ndarray (n_steps,)
    rates : np.ndarray (n_steps, N_total)
    """
    if phi_func is None:
        phi_func = phi_threshold_linear
    if phi_kwargs is None:
        phi_kwargs = {'theta': 0.0, 'r_max': 100.0}

    n_steps = int(T / dt)
    times = np.arange(n_steps) * dt
    rates = np.zeros((n_steps, N_total))

    # Per-neuron time constant vector
    tau_vec = np.ones(N_total) * 0.02
    for cell_type, (s, e) in type_ranges.items():
        if cell_type in tau_dict:
            tau_vec[s:e] = tau_dict[cell_type]

    if r0 is not None:
        rates[0] = r0.copy()

    # Euler integration
    for step in range(1, n_steps):
        t = times[step]
        r = rates[step - 1]
        drive = W @ r + I_ext_func(t, N_total)
        if noise_sigma > 0:
            drive += np.random.normal(0, noise_sigma, N_total)
        drdt = (-r + phi_func(drive, **phi_kwargs)) / tau_vec
        rates[step] = np.maximum(0.0, r + dt * drdt)

    return times, rates


def make_stimulus_func(stim_specs, N_total):
    """
    Build a stimulus function from a list of stimulus specifications.

    Parameters
    ----------
    stim_specs : list of dict
        Each dict has keys: 'indices', 't_on', 't_off', 'amplitude'.
    N_total : int
        Total neuron count.

    Returns
    -------
    callable(t, N_total) → np.ndarray
    """
    def I_ext(t, N):
        stim = np.zeros(N)
        for spec in stim_specs:
            if spec['t_on'] <= t <= spec['t_off']:
                for idx in spec['indices']:
                    stim[idx] = spec['amplitude']
        return stim
    return I_ext


# ============================================================
# analysis.py
# ============================================================

"""
Analysis Tools
==============
Bump decoding, metrics, and stability-sweep routines.
"""

import numpy as np
# from .simulation import simulate_ring_attractor, make_stimulus_func  # inlined above
# from .weights import find_epg_indices  # inlined above


def decode_bump_angle(rates_epg, n_epg):
    """
    Decode bump position as circular mean angle.

    Parameters
    ----------
    rates_epg : np.ndarray (n_epg,)
        EPG firing rates (single time-point).
    n_epg : int
        Number of EPG neurons.

    Returns
    -------
    angle : float
        Decoded angle in radians (−π, π].
    """
    theta = np.arange(n_epg) * (2 * np.pi / n_epg)
    cx = np.sum(rates_epg * np.cos(theta))
    cy = np.sum(rates_epg * np.sin(theta))
    return np.arctan2(cy, cx)


def decode_bump_position(rates, type_range):
    """
    Decode bump position over time via population vector.

    Returns
    -------
    positions : np.ndarray (n_steps,)   – angle in radians
    amplitudes : np.ndarray (n_steps,)  – population vector length
    """
    s, e = type_range
    N = e - s
    theta = np.arange(N) * (2 * np.pi / N)
    r = rates[:, s:e]
    cx = np.sum(r * np.cos(theta), axis=1)
    cy = np.sum(r * np.sin(theta), axis=1)
    return np.arctan2(cy, cx), np.sqrt(cx**2 + cy**2)


def compute_bump_metrics(rates, type_range, t_start=None, t_end=None,
                         times=None):
    """
    Compute ring attractor quality metrics.

    Returns
    -------
    dict with keys: peak_mean_ratio, bump_width_fwhm,
    is_ring_attractor, mean_amplitude, peak_rate, mean_rate.
    """
    s, e = type_range
    N = e - s
    if t_start is not None and times is not None:
        i0 = np.searchsorted(times, t_start)
        i1 = np.searchsorted(times, t_end) if t_end else len(times)
    else:
        i0, i1 = 0, rates.shape[0]

    r = rates[i0:i1, s:e]
    r_mean = np.mean(r, axis=0)
    peak = np.max(r_mean)
    mean = np.mean(r_mean)
    ratio = peak / mean if mean > 0 else 0.0
    fwhm = int(np.sum(r_mean >= peak / 2))

    positions, amplitudes = decode_bump_position(rates[i0:i1],
                                                 (0, N))
    # Fix: decode from sub-slice → type_range is (0, N)
    # Actually need to re-slice to match
    _rates_sub = np.zeros((i1 - i0, N))
    _rates_sub[:, :N] = rates[i0:i1, s:e]
    _, amplitudes = decode_bump_position(_rates_sub, (0, N))
    mean_amp = np.mean(amplitudes)

    return {
        'peak_mean_ratio': ratio,
        'bump_width_fwhm': fwhm,
        'is_ring_attractor': (ratio > 3.0) and (mean_amp > 0.1 * peak * N),
        'mean_amplitude': mean_amp,
        'peak_rate': peak,
        'mean_rate': mean,
    }


def run_stability_sweep(W, N_total, N_E, type_ranges, sim_uids,
                        id_to_type, id_to_pb, config, eb_ring):
    """
    Run a 17-condition sweep (1 baseline + 16 2nd-stim glomeruli).

    Parameters
    ----------
    config : dict
        Must contain: gain, r_max, theta, T, dt, tau, noise,
        stim1_pbs, stim1_t, stim1_amp, stim2_t, stim2_amp.
    eb_ring : list of str
        EB ring glomerulus order (16 entries).

    Returns
    -------
    results : dict
        glomerulus_label → {'rates_epg_final', 'rates_d7_final', ...}
    kymographs : dict
        glomerulus_label → {'times', 'rates_epg', ...}   (full time series)
    """
    phi_kw = {'theta': config['theta'], 'r_max': config['r_max']}
    tau_d = {'EPG': config['tau'], 'Delta7': config['tau']}

    s1_idx = find_epg_indices(sim_uids, id_to_type, id_to_pb, N_E,
                              config['stim1_pbs'])
    sweep_gloms = ['none'] + eb_ring
    results, kymographs = {}, {}

    for s2_pb in sweep_gloms:
        s2_idx = find_epg_indices(sim_uids, id_to_type, id_to_pb, N_E,
                                  [s2_pb])
        stim_specs = [
            {'indices': s1_idx, 't_on': config['stim1_t'][0],
             't_off': config['stim1_t'][1], 'amplitude': config['stim1_amp']},
        ]
        if s2_pb != 'none':
            stim_specs.append(
                {'indices': s2_idx, 't_on': config['stim2_t'][0],
                 't_off': config['stim2_t'][1], 'amplitude': config['stim2_amp']}
            )

        times, rates = simulate_ring_attractor(
            W=W, N_total=N_total, type_ranges=type_ranges,
            tau_dict=tau_d,
            I_ext_func=make_stimulus_func(stim_specs, N_total),
            T=config['T'], dt=config['dt'], phi_kwargs=phi_kw,
            noise_sigma=config['noise'],
        )
        results[s2_pb] = {
            'rates_epg_final': rates[-1, :N_E].copy(),
            'rates_d7_final': rates[-1, N_E:].copy(),
        }
        kymographs[s2_pb] = {
            'times': times,
            'rates_epg': rates[:, :N_E],
            'rates_d7': rates[:, N_E:],
        }

    return results, kymographs


# ============================================================
# visualization.py
# ============================================================

"""
Visualization
=============
Publication-quality plotting for connectome data and simulation results.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# from .weights import BLOCK_SIGNS  # inlined above

# Reusable colour maps
YG_CMAP = LinearSegmentedColormap.from_list('yg', ['#f1c40f', '#27ae60'], N=16)


# ── Connectome block heatmaps ───────────────────────────────────────────

def plot_block_heatmaps(blocks, row_labels, col_labels, suptitle,
                        block_labels=None, vmax=None, figsize=(14, 12)):
    """
    Plot 2×2 grid of E–I connectivity blocks.

    Parameters
    ----------
    blocks : dict  {'E→E', 'E→I', 'I→E', 'I→I'} → DataFrame/ndarray
    row_labels, col_labels : dict → list of str
    suptitle : str
    block_labels : list of 4 str, optional
    vmax : float, optional

    Returns
    -------
    fig, axes
    """
    if block_labels is None:
        block_labels = ['E→E', 'E→I', 'I→E', 'I→I']

    block_names = ['E→E', 'E→I', 'I→E', 'I→I']
    positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
    type_map = {
        'E→E': ('EPG (pre)', 'EPG (post)'),
        'E→I': ('EPG (pre)', 'Δ7 (post)'),
        'I→E': ('Δ7 (pre)', 'EPG (post)'),
        'I→I': ('Δ7 (pre)', 'Δ7 (post)'),
    }
    sign_map = {'E→E': +1, 'I→E': -1, 'E→I': +1, 'I→I': -1}

    signed = {}
    for b in block_names:
        vals = blocks[b].values if hasattr(blocks[b], 'values') else blocks[b]
        signed[b] = vals * sign_map[b]

    if vmax is None:
        vmax = max(np.abs(signed[b]).max() for b in block_names) or 0.01

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    for bname, (r, c), label in zip(block_names, positions, block_labels):
        ax = axes[r, c]
        im = ax.imshow(signed[bname], aspect='auto', cmap='RdBu_r',
                       vmin=-vmax, vmax=vmax, interpolation='none')
        ax.set_title(label, fontsize=13, fontweight='bold')
        rl, cl = row_labels[bname], col_labels[bname]
        ax.set_yticks(range(len(rl)))
        ax.set_yticklabels(rl, fontsize=6)
        ax.set_xticks(range(len(cl)))
        ax.set_xticklabels(cl, fontsize=6, rotation=90)
        ax.set_ylabel(type_map[bname][0])
        ax.set_xlabel(type_map[bname][1])
        ax.grid(False)
        ax.tick_params(length=0)
        plt.colorbar(im, ax=ax, label='syn. strength')
    fig.suptitle(suptitle, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig, axes


# ── Kymographs ──────────────────────────────────────────────────────────

def plot_kymograph_pair(times, rates_fly, rates_aedes, N_E_fly, N_E_aedes,
                        stim_highlights=None, title_suffix='',
                        figsize=(18, 10)):
    """
    Side-by-side EPG + Δ7 kymographs for both species.

    Parameters
    ----------
    times : np.ndarray
    rates_fly, rates_aedes : np.ndarray (n_steps, N_total)
    stim_highlights : list of dict, optional
        Each dict: {'t': (t_on, t_off), 'color': str, 'label': str}

    Returns
    -------
    fig, axes  (2 rows × 2 cols)
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True)

    data = [
        (rates_fly, N_E_fly, 'Drosophila'),
        (rates_aedes, N_E_aedes, 'Aedes'),
    ]
    for row, (rates, N_E, name) in enumerate(data):
        N_total = rates.shape[1]
        for col, (s, e, label) in enumerate([
            (0, N_E, 'EPG'), (N_E, N_total, 'Δ7')
        ]):
            ax = axes[row, col]
            im = ax.imshow(rates[:, s:e].T, aspect='auto', origin='lower',
                           extent=[times[0], times[-1], 0, e - s],
                           cmap='hot', interpolation='none')
            ax.set_ylabel(f'{name} {label}')
            plt.colorbar(im, ax=ax, label='Hz')
            if stim_highlights:
                for sh in stim_highlights:
                    ax.axvspan(*sh['t'], facecolor=sh['color'], alpha=0.12)
                    for t_edge in sh['t']:
                        ax.axvline(t_edge, color=sh['color'], lw=0.8,
                                   ls='--', alpha=0.5)
            if row == 0:
                ax.set_title(f'{label} Activity {title_suffix}',
                             fontweight='bold')

    axes[-1, 0].set_xlabel('Time (s)')
    axes[-1, 1].set_xlabel('Time (s)')
    plt.tight_layout()
    return fig, axes


# ── EPG activity profiles ───────────────────────────────────────────────

def plot_epg_profiles(sweep_results, eb_ring, config, species_name,
                      ax=None):
    """
    Plot EPG activity at T=end for a sweep: baseline (black)
    + each 2nd-stim (yellow→green gradient).

    Parameters
    ----------
    sweep_results : dict
        glomerulus → {'rates_epg_final': array}
    eb_ring : list of str
    config : dict
    species_name : str
    ax : matplotlib Axes, optional

    Returns
    -------
    ax
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 5))

    n = len(eb_ring)
    colors = [YG_CMAP(i / max(n - 1, 1)) for i in range(n)]

    for gi, glom in enumerate(eb_ring):
        if glom in sweep_results:
            ax.plot(sweep_results[glom]['rates_epg_final'], '-',
                    color=colors[gi], alpha=0.6, lw=1.2, label=glom)

    if 'none' in sweep_results:
        ax.plot(sweep_results['none']['rates_epg_final'], 'k-',
                lw=2.5, alpha=0.9, label='baseline', zorder=10)

    ax.set_xlabel('EPG neuron index (EB order)')
    ax.set_ylabel('Rate at T=end (Hz)')
    ax.set_title(f'{species_name} — EPG at T={config["T"]}s',
                 fontweight='bold')
    ax.grid(True, alpha=0.2)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[-1:] + handles[:-1], labels[-1:] + labels[:-1],
                  fontsize=7, ncol=5, loc='upper left', framealpha=0.8)
    return ax


# ── Stability sweep grid ────────────────────────────────────────────────

def plot_stability_grid(kymographs_fly, kymographs_aedes, config,
                        eb_ring, N_E_fly, N_E_aedes,
                        figsize_per_row=2.5):
    """
    17-row sweep kymograph: baseline + each 2nd-stim perturbation.

    Returns
    -------
    fig, axes
    """
    sweep_gloms = ['none'] + eb_ring
    n_rows = len(sweep_gloms)
    fig, axes = plt.subplots(n_rows, 2, figsize=(18, figsize_per_row * n_rows),
                             sharex=True)

    stim_hl = [
        {'t': config['stim1_t'], 'color': 'lime'},
        {'t': config['stim2_t'], 'color': 'cyan'},
    ]

    for row, glom in enumerate(sweep_gloms):
        label = 'baseline' if glom == 'none' else f'2nd={glom}'
        for col, (kymo_dict, N_E, name) in enumerate([
            (kymographs_fly, N_E_fly, 'Fly'),
            (kymographs_aedes, N_E_aedes, 'Aedes'),
        ]):
            ax = axes[row, col]
            k = kymo_dict[glom]
            im = ax.imshow(k['rates_epg'].T, aspect='auto', origin='lower',
                           extent=[k['times'][0], k['times'][-1], 0, N_E],
                           cmap='hot', interpolation='none')
            for sh in stim_hl:
                ax.axvspan(*sh['t'], facecolor=sh['color'], alpha=0.1)
            if col == 0:
                ax.set_ylabel(label, fontsize=9, fontweight='bold')
            if row == 0:
                ax.set_title(f'{name} EPG', fontweight='bold')
            if row == n_rows - 1:
                ax.set_xlabel('Time (s)')
            ax.grid(False)
            ax.tick_params(length=0)

    plt.suptitle(
        f'Stability Sweep — gain={config["gain"]}, r_max={config["r_max"]}, '
        f'θ={config["theta"]}',
        fontsize=14, fontweight='bold', y=1.001
    )
    plt.tight_layout()
    return fig, axes


# ── Full stim matrix grid ───────────────────────────────────────────────

def plot_full_sweep_grid(full_profiles, eb_ring, config, N_E_fly, N_E_aedes,
                         figsize_per_row=3):
    """
    16-row × 2-col grid: each row = different 1st stim.

    Parameters
    ----------
    full_profiles : dict
        species → stim1_pb → stim2_pb → {'epg': array}
    """
    n_init = len(eb_ring)
    colors = [YG_CMAP(i / max(n_init - 1, 1)) for i in range(n_init)]

    fig, axes = plt.subplots(n_init, 2,
                             figsize=(18, figsize_per_row * n_init),
                             sharex=True)

    for row, s1_pb in enumerate(eb_ring):
        for col, (species, N_E) in enumerate([
            ('fly', N_E_fly), ('aedes', N_E_aedes),
        ]):
            ax = axes[row, col]
            name = 'Drosophila' if species == 'fly' else 'Aedes'
            profiles = full_profiles[species].get(s1_pb, {})

            for gi, s2_pb in enumerate(eb_ring):
                if s2_pb in profiles:
                    ax.plot(profiles[s2_pb]['epg'], '-',
                            color=colors[gi], alpha=0.6, lw=1.2,
                            label=s2_pb)
            if 'none' in profiles:
                ax.plot(profiles['none']['epg'], 'k-',
                        lw=2.5, alpha=0.9, label='baseline', zorder=10)

            ax.grid(True, alpha=0.2)
            ax.tick_params(length=0)
            if col == 0:
                ax.set_ylabel(f'1st={s1_pb}', fontsize=10, fontweight='bold')
            if row == 0:
                ax.set_title(f'{name} EPG', fontsize=13, fontweight='bold')
            if row == n_init - 1:
                ax.set_xlabel('EPG neuron index (EB order)')
            if row == 0:
                h, l = ax.get_legend_handles_labels()
                if h:
                    ax.legend(h[-1:] + h[:-1], l[-1:] + l[:-1],
                              fontsize=5, ncol=5, loc='upper right',
                              framealpha=0.8)

    plt.suptitle(
        f'Full Sweep — T={config["T"]}s (gain={config["gain"]}, '
        f'r_max={config["r_max"]}, θ={config["theta"]})',
        fontsize=14, fontweight='bold', y=1.001
    )
    plt.tight_layout()
    return fig, axes


# ── Activation function comparison ──────────────────────────────────────

def plot_activation_comparison(r_max=100.0, theta=0.0):
    """Plot threshold-linear vs sigmoid comparison (3 panels)."""
# from .activation import phi_threshold_linear, phi_sigmoid  # inlined above

    x = np.linspace(-50, 200, 1000)
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Panel 1: threshold-linear variants
    ax = axes[0]
    ax.set_title('Threshold-Linear', fontweight='bold')
    for th in [0, 2, 5, 10]:
        ax.plot(x, np.clip(x - th, 0, r_max), '--', alpha=0.5, lw=1.5,
                label=f'θ={th}')
    ax.plot(x, np.clip(x - theta, 0, r_max), 'r-', lw=3,
            label=f'current (θ={theta})')
    ax.set_xlabel('Drive')
    ax.set_ylabel('Rate (Hz)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(-5, r_max * 1.2)

    # Panel 2: sigmoid variants
    ax = axes[1]
    ax.set_title('Sigmoid', fontweight='bold')
    for beta in [0.02, 0.05, 0.1, 0.2]:
        ax.plot(x, r_max / (1 + np.exp(-beta * (x - 50))), lw=2,
                alpha=0.8, label=f'β={beta}')
    ax.set_xlabel('Drive')
    ax.set_ylabel('Rate (Hz)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(-5, r_max * 1.2)

    # Panel 3: comparison
    ax = axes[2]
    ax.set_title('Comparison', fontweight='bold')
    y_tl = np.clip(x - theta, 0, r_max)
    y_sig = r_max / (1 + np.exp(-0.05 * (x - 30)))
    ax.plot(x, y_tl, 'r-', lw=2.5, label='Threshold-linear', alpha=0.8)
    ax.plot(x, y_sig, '#2ecc71', lw=2.5, label='Sigmoid (β=0.05)', alpha=0.8)
    ax.fill_between(x, y_tl, y_sig, alpha=0.1, color='blue',
                    where=(x > 0) & (x < 150))
    ax.set_xlabel('Drive')
    ax.set_ylabel('Rate (Hz)')
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.2)
    ax.set_ylim(-5, r_max * 1.2)

    plt.suptitle('Activation Function φ(x): Drive → Firing Rate',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig, axes


print('All cx_model functions inlined successfully.')


## 1. Configuration

All tunable parameters are defined here. Edit this cell and re-run the
notebook to explore different regimes.


In [ ]:
# ======================================================================
# ⭐ SIMULATION PARAMETERS — edit these to tune the model
# ======================================================================

CONFIG = dict(
    # Network
    gain        = 20.0,    # synaptic weight scaling
    r_max       = 100.0,   # activation ceiling (Hz)
    theta       = 0.0,     # activation threshold
    tau         = 0.020,   # membrane time constant (s)
    noise       = 0,       # noise sigma (0 = deterministic)
    syn_thresh  = 2,       # min synapse count for connectivity

    # Timing
    T           = 8,       # total simulation time (s)
    dt          = 0.01,    # time step (s)

    # Stimulus 1 (initial bump)
    stim1_pbs   = ['L4'],  # PB glomerulus
    stim1_t     = (0.5, 1.0),
    stim1_amp   = 100.0,

    # Stimulus 2 (perturbation)
    stim2_pbs   = ['L1'],
    stim2_t     = (4.5, 5.0),
    stim2_amp   = 100.0,
)

# Print for verification
print('Simulation configuration:')
for k, v in CONFIG.items():
    print(f'  {k:14s} = {v}')


## 2. Load & Inspect Connectome

Load EPG–Δ7 connectivity with synapse-count thresholding.


In [ ]:
# Load with threshold
data = load_connectome(DATA_PATH,
                       syn_threshold=CONFIG['syn_thresh'])

# Extract per-species data
fly_sp  = get_species_data(data, 'fly')
aedes_sp = get_species_data(data, 'aedes')

print(f"\nFly EPGs:   {sum(1 for v in fly_sp['id_to_type'].values() if v=='EPG')}")
print(f"Aedes EPGs: {sum(1 for v in aedes_sp['id_to_type'].values() if v=='EPG')}")


In [ ]:
# 4-block connectivity heatmaps (PB glomerulus order)
for sp, name, blabels in [
    (fly_sp, 'Drosophila',
     ['E→E (+local)', 'E→I (+opposite)', 'I→E (−local)', 'I→I (−uniform)']),
    (aedes_sp, 'Aedes',
     ['E→E (+uniform)', 'E→I (+opposite)', 'I→E (−local)', 'I→I (−antiphase)']),
]:
    blocks, rl, cl = extract_4blocks(
        sp['matrix_norm'], sp['sorted_ids_pb'],
        sp['id_to_type'], sp['id_to_pb'],
        i_order=sp['delta7_order'],
    )
    plot_block_heatmaps(blocks, rl, cl,
                        f'{name} — PB Order (syn ≥ {CONFIG["syn_thresh"]})',
                        block_labels=blabels, vmax=0.1)
    plt.show()


## 3. Single Simulation

Build weight matrices and run the ring attractor for both species.


In [ ]:
# Build weight matrices
W_fly, uids_fly, N_E_fly, tr_fly = build_weight_matrix(
    fly_sp['matrix_norm'], fly_sp['sorted_ids_eb'],
    fly_sp['id_to_type'], fly_sp['id_to_pb'],
    CONFIG['gain'], fly_sp['delta7_order'],
)
W_aedes, uids_aedes, N_E_aedes, tr_aedes = build_weight_matrix(
    aedes_sp['matrix_norm'], aedes_sp['sorted_ids_eb'],
    aedes_sp['id_to_type'], aedes_sp['id_to_pb'],
    CONFIG['gain'], aedes_sp['delta7_order'],
)

phi_kw = {'theta': CONFIG['theta'], 'r_max': CONFIG['r_max']}
tau_d  = {'EPG': CONFIG['tau'], 'Delta7': CONFIG['tau']}

# Run both species
results = {}
for label, W, N_tot, N_E, tr, uids, sp in [
    ('fly', W_fly, len(uids_fly), N_E_fly, tr_fly, uids_fly, fly_sp),
    ('aedes', W_aedes, len(uids_aedes), N_E_aedes, tr_aedes, uids_aedes, aedes_sp),
]:
    s1_idx = find_epg_indices(uids, sp['id_to_type'], sp['id_to_pb'],
                              N_E, CONFIG['stim1_pbs'])
    s2_idx = find_epg_indices(uids, sp['id_to_type'], sp['id_to_pb'],
                              N_E, CONFIG['stim2_pbs'])
    stim = make_stimulus_func([
        {'indices': s1_idx, 't_on': CONFIG['stim1_t'][0],
         't_off': CONFIG['stim1_t'][1], 'amplitude': CONFIG['stim1_amp']},
        {'indices': s2_idx, 't_on': CONFIG['stim2_t'][0],
         't_off': CONFIG['stim2_t'][1], 'amplitude': CONFIG['stim2_amp']},
    ], N_tot)

    t, r = simulate_ring_attractor(
        W, N_tot, tr, tau_d, stim,
        T=CONFIG['T'], dt=CONFIG['dt'],
        phi_kwargs=phi_kw, noise_sigma=CONFIG['noise'],
    )
    results[label] = {'times': t, 'rates': r}

    m = compute_bump_metrics(r, tr['EPG'], t_start=CONFIG['T']/2, times=t)
    name = 'Drosophila' if label == 'fly' else 'Aedes'
    print(f"{name}: peak/mean={m['peak_mean_ratio']:.2f}, "
          f"FWHM={m['bump_width_fwhm']}, "
          f"ring_attractor={m['is_ring_attractor']}")


In [ ]:
# Kymographs: EPG + Δ7 for both species
stim_hl = [
    {'t': CONFIG['stim1_t'], 'color': 'lime', 'label': 'Stim 1'},
    {'t': CONFIG['stim2_t'], 'color': 'cyan', 'label': 'Stim 2'},
]
plot_kymograph_pair(
    results['fly']['times'],
    results['fly']['rates'], results['aedes']['rates'],
    N_E_fly, N_E_aedes,
    stim_highlights=stim_hl,
    title_suffix=f'(gain={CONFIG["gain"]})'
)
plt.show()


## 4. Stability Sweep

Test whether the heading bump can be shifted by a 2nd stimulus at each
of the 16 PB glomeruli. Baseline (no 2nd stim) included for reference.


In [ ]:
# Run 17-condition sweep for both species
sweep_fly, kymo_fly = run_stability_sweep(
    W_fly, len(uids_fly), N_E_fly, tr_fly,
    uids_fly, fly_sp['id_to_type'], fly_sp['id_to_pb'],
    CONFIG, EB_RING_ORDER,
)
sweep_aedes, kymo_aedes = run_stability_sweep(
    W_aedes, len(uids_aedes), N_E_aedes, tr_aedes,
    uids_aedes, aedes_sp['id_to_type'], aedes_sp['id_to_pb'],
    CONFIG, EB_RING_ORDER,
)
print('Stability sweep complete.')


In [ ]:
plot_stability_grid(kymo_fly, kymo_aedes, CONFIG,
                    EB_RING_ORDER, N_E_fly, N_E_aedes)
plt.show()


## 5. Bump Shift Analysis

EPG activity at T=end: baseline (black) vs each 2nd stimulus
(yellow → green gradient).


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))
for ax, (sweep, name) in zip(axes, [
    (sweep_fly, 'Drosophila'), (sweep_aedes, 'Aedes')
]):
    plot_epg_profiles(sweep, EB_RING_ORDER, CONFIG, name, ax=ax)
plt.suptitle(f'Bump Shift Analysis — T={CONFIG["T"]}s '
             f'(gain={CONFIG["gain"]}, r_max={CONFIG["r_max"]})',
             fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Full Stimulus Matrix

Sweep **all** initial × perturbation combinations (16 × 17 × 2 = 544 sims).
Each row = different 1st stimulus; black = baseline, colours = 2nd stim.


In [ ]:
# Full sweep: all stim1 × all stim2, both species
full_profiles = {'fly': {}, 'aedes': {}}
n_total_sims = len(EB_RING_ORDER) * 17 * 2
sim_count = 0

for s1_pb in EB_RING_ORDER:
    for label, W, N_tot, N_E, tr, uids, sp in [
        ('fly', W_fly, len(uids_fly), N_E_fly, tr_fly, uids_fly, fly_sp),
        ('aedes', W_aedes, len(uids_aedes), N_E_aedes, tr_aedes, uids_aedes, aedes_sp),
    ]:
        s1_idx = find_epg_indices(uids, sp['id_to_type'], sp['id_to_pb'], N_E, [s1_pb])
        if s1_pb not in full_profiles[label]:
            full_profiles[label][s1_pb] = {}

        for s2_pb in ['none'] + list(EB_RING_ORDER):
            s2_idx = find_epg_indices(uids, sp['id_to_type'], sp['id_to_pb'], N_E, [s2_pb])
            specs = [{'indices': s1_idx, 't_on': CONFIG['stim1_t'][0],
                      't_off': CONFIG['stim1_t'][1], 'amplitude': CONFIG['stim1_amp']}]
            if s2_pb != 'none':
                specs.append({'indices': s2_idx, 't_on': CONFIG['stim2_t'][0],
                              't_off': CONFIG['stim2_t'][1], 'amplitude': CONFIG['stim2_amp']})

            t, r = simulate_ring_attractor(
                W, N_tot, tr, {'EPG': CONFIG['tau'], 'Delta7': CONFIG['tau']},
                make_stimulus_func(specs, N_tot),
                T=CONFIG['T'], dt=CONFIG['dt'],
                phi_kwargs={'theta': CONFIG['theta'], 'r_max': CONFIG['r_max']},
                noise_sigma=CONFIG['noise'],
            )
            full_profiles[label][s1_pb][s2_pb] = {'epg': r[-1, :N_E].copy()}
            sim_count += 1

    print(f'  stim1={s1_pb} done ({sim_count/n_total_sims*100:.0f}%)')

print(f'\nAll {sim_count} simulations complete.')


In [ ]:
plot_full_sweep_grid(full_profiles, EB_RING_ORDER, CONFIG,
                     N_E_fly, N_E_aedes)
plt.show()


## Appendix: Activation Function Comparison

Threshold-linear (current) vs sigmoid (alternative).


In [ ]:
plot_activation_comparison(r_max=CONFIG['r_max'], theta=CONFIG['theta'])
plt.show()
